# 🚀 Industry-Grade News Article Classification with DistilBERT

**Portfolio-ready deep learning project** — fine-tunes DistilBERT on the AG News dataset
(40,000 training samples) for 4-class text classification.

| Stage | Description |
|---|---|
| Dataset | AG News (HuggingFace) — 40k train / 7.6k test |
| Model | `distilbert-base-uncased` + classification head |
| Training | 3 epochs, AdamW + linear warm-up, fp16 on GPU |
| Metrics | Accuracy, weighted F1, confusion matrix |

> **Runtime**: Go to *Runtime → Change runtime type → T4 GPU* before running.

## 1 · Install Dependencies

In [ ]:
!pip install -q datasets transformers torch scikit-learn

## 2 · Clone / Mount Repository

In [ ]:
# Option A — clone repo directly
!git clone https://github.com/ankit3010droid/chat1.git
%cd chat1

# Option B — mount Google Drive (if you uploaded the repo there)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/chat1

## 3 · Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4 · Download & Preview Dataset

In [ ]:
from src.preprocess import load_ag_news
from src.utils import LABEL_NAMES

train_hf, test_hf = load_ag_news(train_size=40_000, seed=42)
print(f'Train samples : {len(train_hf):,}')
print(f'Test  samples : {len(test_hf):,}')
print(f'Labels        : {LABEL_NAMES}')
print('\nSample entry:')
print(train_hf[0])

## 5 · Preprocess & Build Datasets

In [ ]:
from src.preprocess import load_tokenizer, build_dataset

tokenizer = load_tokenizer()
train_dataset = build_dataset(train_hf, tokenizer)
test_dataset  = build_dataset(test_hf,  tokenizer)

# Inspect a single encoded sample
sample = train_dataset[0]
print('input_ids shape  :', sample['input_ids'].shape)
print('attention_mask   :', sample['attention_mask'].shape)
print('label            :', sample['labels'].item(), '->', LABEL_NAMES[sample['labels'].item()])

## 6 · Train the Model

In [ ]:
from src.train import train

# Trains for 3 epochs and saves the model to models/distilbert.pt
model = train(train_size=40_000, seed=42)

## 7 · Evaluate on Test Set

In [ ]:
from src.evaluate import evaluate

metrics = evaluate()
print(f"\nFinal Accuracy : {metrics['accuracy']:.4f}")
print(f"Final F1 Score : {metrics['f1']:.4f}")

## 8 · Inference on Custom Input

In [ ]:
from src.inference import predict

custom_texts = [
    'Apple unveils new MacBook with M4 chip and AI-powered features.',
    'Brazil wins the FIFA World Cup after dramatic penalty shootout.',
    'IMF forecasts global GDP growth of 3.2 percent for next year.',
    'NATO allies pledge support for Ukraine at emergency summit.',
]

print('Predictions:')
print('-' * 60)
for text in custom_texts:
    label = predict(text)
    print(f'  [{label:10s}]  {text[:65]}...')

## 9 · Download the Trained Model

Run the cell below to zip and download the saved model weights from Colab to your machine.

In [ ]:
import shutil
shutil.make_archive('distilbert_model', 'zip', 'models/distilbert')

from google.colab import files
files.download('distilbert_model.zip')